In [1]:
import os
import json
import time
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    DecisionTreeClassifier, RandomForestClassifier,
    LogisticRegression, 
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window

import pandas as pd

### Configuration & Spark session

In [2]:
PROJECT_ROOT = Path(
    r"D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics"
).resolve()
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "output"))
FEATURE_PARQUET_PATH = str(OUTPUT_DIR / "final_feature_dataset.parquet")
FEATURE_METADATA_PATH = str(OUTPUT_DIR / "feature_metadata.json")
BEST_MODEL_PATH = str(OUTPUT_DIR / "best_model")

N_CORES = int(os.environ.get("SPARK_CORES", "8"))
CV_FOLDS = int(os.environ.get("CV_FOLDS", "3"))
SEED = 42

spark = (
    SparkSession.builder
    .appName("BusRoute_05_MachineLearning")
    .master(f"local[{N_CORES}]")
    .config("spark.sql.shuffle.partitions", str(N_CORES * 2))
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "6g"))
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI     :", spark.sparkContext.uiWebUrl)

Spark version: 3.5.8
Spark UI     : http://DESKTOP-P7PE4MO:4040


### Load engineered dataset and its feature-role manifest

In [3]:
schedule = spark.read.parquet(FEATURE_PARQUET_PATH).cache()
BASE_ROW_COUNT = schedule.count()

with open(FEATURE_METADATA_PATH) as f:
    feature_metadata = json.load(f)

print("="*60)
print("Dataset Loaded")
print("="*60)
print("Rows    :", f"{BASE_ROW_COUNT:,}")
print("Columns :", len(schedule.columns))
print()
print("Feature metadata from Notebook 04:")
print(json.dumps(feature_metadata, indent=2))

assert BASE_ROW_COUNT > 0, "final_feature_dataset.parquet loaded with zero rows."

Dataset Loaded
Rows    : 926,481
Columns : 32

Feature metadata from Notebook 04:
{
  "target_column": "route_popularity",
  "leakage_columns": [
    "daily_journeys"
  ],
  "removed_leakage_features_at_source": [
    "service_category"
  ],
  "id_columns": [
    "source_file",
    "vehicle_journey_code",
    "stop_point_ref",
    "line_ref",
    "line_name",
    "service_ref",
    "journey_pattern_ref",
    "stop_name",
    "operator_ref"
  ],
  "timestamp_columns": [
    "scheduled_ts"
  ],
  "candidate_feature_columns": [
    "fare_publication_count",
    "fare_publication_level",
    "fare_status",
    "hour_of_day",
    "is_first_stop",
    "is_peak_hour",
    "journey_type",
    "operator_routes",
    "operator_size",
    "operator_trip_count",
    "operator_workload",
    "route_complexity",
    "scheduled_time",
    "stop_activity",
    "stop_busyness",
    "stop_position",
    "stop_progress_pct",
    "stop_sequence",
    "total_stops",
    "unique_stops"
  ],
  "dropped_const

## Pre-training validation

In [4]:
TARGET_COLUMN = feature_metadata["target_column"]
LEAKAGE_COLUMNS = feature_metadata["leakage_columns"]
ID_COLUMNS = feature_metadata["id_columns"]
TIMESTAMP_COLUMNS = feature_metadata["timestamp_columns"]

candidate_features = [
    c for c in feature_metadata["candidate_feature_columns"]
    if c in schedule.columns
]
missing_from_df = set(feature_metadata["candidate_feature_columns"]) - set(schedule.columns)
if missing_from_df:
    print(f"Note: manifest lists columns not present in this parquet file (already dropped?): {missing_from_df}")

DISPLAY_ID_COLUMNS = [c for c in ["line_ref", "vehicle_journey_code", "stop_point_ref"]
                       if c in schedule.columns and c not in candidate_features]

model_df = schedule.select([TARGET_COLUMN] + DISPLAY_ID_COLUMNS + candidate_features)

print(f"Target column     : {TARGET_COLUMN}")
print(f"Excluded (leakage): {LEAKAGE_COLUMNS}")
print(f"Excluded (id)     : {ID_COLUMNS}")
print(f"Excluded (ts)     : {TIMESTAMP_COLUMNS}")
print(f"\nCandidate features ({len(candidate_features)}): {candidate_features}")

assert not (set(candidate_features) & set(LEAKAGE_COLUMNS)), \
    "A leakage column slipped into the feature set -- check the manifest / this cell's filtering logic."
assert "service_category" not in candidate_features, (
    "service_category is target-derived (built from route_popularity)"
)

Target column     : route_popularity
Excluded (leakage): ['daily_journeys']
Excluded (id)     : ['source_file', 'vehicle_journey_code', 'stop_point_ref', 'line_ref', 'line_name', 'service_ref', 'journey_pattern_ref', 'stop_name', 'operator_ref']
Excluded (ts)     : ['scheduled_ts']

Candidate features (20): ['fare_publication_count', 'fare_publication_level', 'fare_status', 'hour_of_day', 'is_first_stop', 'is_peak_hour', 'journey_type', 'operator_routes', 'operator_size', 'operator_trip_count', 'operator_workload', 'route_complexity', 'scheduled_time', 'stop_activity', 'stop_busyness', 'stop_position', 'stop_progress_pct', 'stop_sequence', 'total_stops', 'unique_stops']


In [5]:
null_report = model_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in model_df.columns
]).collect()[0].asDict()

null_df = pd.DataFrame(
    [(c, n, round(100 * n / BASE_ROW_COUNT, 2)) for c, n in null_report.items() if n > 0],
    columns=["Column", "Null Count", "Null %"],
)
if len(null_df):
    print(null_df.sort_values("Null Count", ascending=False).to_string(index=False))
    high_null_cols = null_df[null_df["Null %"] > 50]["Column"].tolist()
    assert not high_null_cols, f"Column(s) with >50% nulls should be re-examined before modelling: {high_null_cols}"
else:
    print("No missing values in the modelling columns.")

No missing values in the modelling columns.


In [6]:
numeric_candidates = [f.name for f in model_df.schema.fields
                       if f.name in candidate_features and isinstance(f.dataType, NumericType)]
categorical_candidates = [c for c in candidate_features if c not in numeric_candidates]

variance_report = model_df.select([F.variance(F.col(c)).alias(c) for c in numeric_candidates]).collect()[0].asDict()
zero_variance = [c for c, v in variance_report.items() if v is None or v == 0]

distinct_report = model_df.select([F.countDistinct(F.col(c)).alias(c) for c in categorical_candidates]).collect()[0].asDict()
constant_categorical = [c for c, n in distinct_report.items() if n <= 1]

print("Zero-variance numeric features:", zero_variance or "none")
print("Constant categorical features :", constant_categorical or "none")

for c in zero_variance + constant_categorical:
    candidate_features.remove(c)
    if c in numeric_candidates:
        numeric_candidates.remove(c)
    if c in categorical_candidates:
        categorical_candidates.remove(c)

print(f"\nNumeric features    ({len(numeric_candidates)}): {numeric_candidates}")
print(f"Categorical features ({len(categorical_candidates)}): {categorical_candidates}")

Zero-variance numeric features: none
Constant categorical features : none

Numeric features    (11): ['fare_publication_count', 'hour_of_day', 'is_first_stop', 'is_peak_hour', 'operator_routes', 'operator_trip_count', 'stop_activity', 'stop_progress_pct', 'stop_sequence', 'total_stops', 'unique_stops']
Categorical features (9): ['fare_publication_level', 'fare_status', 'journey_type', 'operator_size', 'operator_workload', 'route_complexity', 'scheduled_time', 'stop_busyness', 'stop_position']


### Correlation among numeric features 

In [7]:
from itertools import combinations

corr_rows = []
for a, b in combinations(numeric_candidates, 2):
    corr_rows.append((a, b, model_df.stat.corr(a, b)))
corr_df = pd.DataFrame(corr_rows, columns=["Feature A", "Feature B", "Pearson r"]).sort_values(
    "Pearson r", key=abs, ascending=False
)
print(corr_df.to_string(index=False))

highly_correlated = corr_df[corr_df["Pearson r"].abs() >= 0.95]
if len(highly_correlated):
    print("\n! Highly correlated pairs (|r| >= 0.95) -- consider dropping one of each pair:")
    print(highly_correlated.to_string(index=False))
else:
    print("\nNo pair of numeric features exceeds |r| = 0.95.")

             Feature A           Feature B  Pearson r
           total_stops        unique_stops   0.876777
     stop_progress_pct       stop_sequence   0.748699
fare_publication_count operator_trip_count   0.701493
         stop_sequence         total_stops   0.567833
         stop_sequence        unique_stops   0.498054
       operator_routes operator_trip_count   0.487769
fare_publication_count     operator_routes   0.374470
fare_publication_count       stop_activity   0.272244
         is_first_stop   stop_progress_pct  -0.266280
   operator_trip_count       stop_activity   0.246966
         is_first_stop       stop_sequence  -0.218035
         stop_activity        unique_stops  -0.197390
       operator_routes        unique_stops   0.180851
         stop_activity         total_stops  -0.162311
       operator_routes         total_stops   0.148229
         is_first_stop         total_stops  -0.094317
       operator_routes       stop_activity   0.088161
       operator_routes      

In [8]:
class_counts = model_df.groupBy(TARGET_COLUMN).count().orderBy(F.desc("count")).toPandas()
class_counts["pct"] = (100 * class_counts["count"] / BASE_ROW_COUNT).round(2)
print(class_counts.to_string(index=False))

imbalance_ratio = class_counts["count"].max() / class_counts["count"].min()
print(f"\nImbalance ratio (largest class / smallest class): {imbalance_ratio:.2f}")
if imbalance_ratio > 3:
    print("Ratio > 3 -- class weighting will be applied where the estimator supports it.")

route_popularity  count   pct
            High 325492 35.13
             Low 304386 32.85
          Medium 296603 32.01

Imbalance ratio (largest class / smallest class): 1.10


##  Train/test split

In [9]:
naive_train, naive_test = model_df.randomSplit([0.8, 0.2], seed=SEED)

naive_train_routes = naive_train.select("line_ref").distinct()
naive_test_routes = naive_test.select("line_ref").distinct()
naive_overlap = naive_train_routes.intersect(naive_test_routes).count()

n_train_routes_naive = naive_train_routes.count()
n_test_routes_naive = naive_test_routes.count()

print("Row-level randomSplit (ORIGINAL, leaky) diagnostic:")
print(f"  Train routes: {n_train_routes_naive:,}")
print(f"  Test routes : {n_test_routes_naive:,}")
print(f"  Routes appearing in BOTH train and test: {naive_overlap:,} "
      f"({100*naive_overlap/n_test_routes_naive:.1f}% of test routes were already seen in training)")

del naive_train, naive_test  


Row-level randomSplit (ORIGINAL, leaky) diagnostic:
  Train routes: 209
  Test routes : 209
  Routes appearing in BOTH train and test: 209 (100.0% of test routes were already seen in training)


In [10]:
route_labels = model_df.select("line_ref", TARGET_COLUMN).distinct().coalesce(1).cache()
dup_routes = route_labels.groupBy("line_ref").count().filter(F.col("count") > 1).count()
assert dup_routes == 0, "Some routes map to more than one route_popularity label -- investigate before splitting."

class_window = Window.partitionBy(TARGET_COLUMN).orderBy(F.rand(seed=SEED))
size_window = Window.partitionBy(TARGET_COLUMN)

route_split = (
    route_labels
    .withColumn("rank_in_class", F.row_number().over(class_window))
    .withColumn("class_size", F.count("*").over(size_window))
    .withColumn("split", F.when(F.col("rank_in_class") <= F.col("class_size") * 0.8, "train").otherwise("test"))
    .select("line_ref", "split")
)

print("Routes per class assigned to train/test (stratified):")
route_split.join(route_labels, on="line_ref").groupBy(TARGET_COLUMN, "split").count().orderBy(TARGET_COLUMN, "split").show()

model_df_split = model_df.join(F.broadcast(route_split), on="line_ref", how="left")

train_df = model_df_split.filter(F.col("split") == "train").drop("split")
test_df = model_df_split.filter(F.col("split") == "test").drop("split")
train_df = train_df.cache()
test_df = test_df.cache()

n_train, n_test = train_df.count(), test_df.count()
print(f"Train rows: {n_train:,} ({100*n_train/BASE_ROW_COUNT:.1f}%)")
print(f"Test rows : {n_test:,} ({100*n_test/BASE_ROW_COUNT:.1f}%)")
assert n_train + n_test == BASE_ROW_COUNT

n_train_routes = train_df.select("line_ref").distinct().count()
n_test_routes = test_df.select("line_ref").distinct().count()
route_overlap = (
    train_df.select("line_ref").distinct()
    .intersect(test_df.select("line_ref").distinct())
    .count()
)
assert route_overlap == 0, "Route-level leakage still present -- a line_ref appears in both train and test!"

print("\nTrain class distribution:")
train_df.groupBy(TARGET_COLUMN).count().orderBy(F.desc("count")).show()
print("Test class distribution:")
test_df.groupBy(TARGET_COLUMN).count().orderBy(F.desc("count")).show()


Routes per class assigned to train/test (stratified):
+----------------+-----+-----+
|route_popularity|split|count|
+----------------+-----+-----+
|            High| test|    6|
|            High|train|   21|
|             Low| test|   30|
|             Low|train|  116|
|          Medium| test|    8|
|          Medium|train|   28|
+----------------+-----+-----+

Train rows: 732,926 (79.1%)
Test rows : 193,555 (20.9%)

Train class distribution:
+----------------+------+
|route_popularity| count|
+----------------+------+
|            High|256919|
|          Medium|238112|
|             Low|237895|
+----------------+------+

Test class distribution:
+----------------+-----+
|route_popularity|count|
+----------------+-----+
|            High|68573|
|             Low|66491|
|          Medium|58491|
+----------------+-----+



In [11]:
for _stale_var in ["fitted_models", "comparison_df", "predictions_by_model", "best_model", "best_model_name"]:
    if _stale_var in globals():
        del globals()[_stale_var]
print("Cleared any previous run's trained models / comparison table.")
print("You must now run every cell below, in order, before saving.")


Cleared any previous run's trained models / comparison table.
You must now run every cell below, in order, before saving.


In [12]:
train_class_counts = train_df.groupBy(TARGET_COLUMN).count().collect()
n_classes = len(train_class_counts)
total = sum(r["count"] for r in train_class_counts)
class_weight_map = {r[TARGET_COLUMN]: total / (n_classes * r["count"]) for r in train_class_counts}
print("Class weights (inverse frequency):", class_weight_map)

weight_expr = F.create_map([F.lit(x) for pair in class_weight_map.items() for x in pair])
train_df = train_df.withColumn("class_weight", weight_expr[F.col(TARGET_COLUMN)])
test_df = test_df.withColumn("class_weight", F.lit(1.0))  # weighting only applies at training time

train_df = train_df.cache()
train_df.select(TARGET_COLUMN, "class_weight").distinct().show()

Class weights (inverse frequency): {'High': 0.9509170854108363, 'Medium': 1.02602416789858, 'Low': 1.0269600734217477}
+----------------+------------------+
|route_popularity|      class_weight|
+----------------+------------------+
|            High|0.9509170854108363|
|          Medium|  1.02602416789858|
|             Low|1.0269600734217477|
+----------------+------------------+



## Route-stratified CV folds 

In [13]:
route_fold_window = Window.partitionBy(TARGET_COLUMN).orderBy(F.rand(seed=SEED))

train_routes_for_folds = (
    train_df.select("line_ref", TARGET_COLUMN).distinct().coalesce(1)
)
route_folds = (
    train_routes_for_folds
    .withColumn("rn", F.row_number().over(route_fold_window))
    .withColumn("cv_fold", ((F.col("rn") - 1) % CV_FOLDS).cast("int"))
    .select("line_ref", "cv_fold")
)

train_df = train_df.join(F.broadcast(route_folds), on="line_ref", how="left")
train_df = train_df.cache()

print("Rows per CV fold (should be roughly even):")
train_df.groupBy("cv_fold").count().orderBy("cv_fold").show()

# Verify every route maps to exactly ONE fold (no route split across folds)
fold_conflicts = (
    train_df.select("line_ref", "cv_fold").distinct()
    .groupBy("line_ref").count().filter(F.col("count") > 1).count()
)
assert fold_conflicts == 0, "A route is assigned to more than one CV fold -- check the join above."
print("Verified: every route belongs to exactly one CV fold (route-clean hyperparameter search).")


Rows per CV fold (should be roughly even):
+-------+------+
|cv_fold| count|
+-------+------+
|      0|239667|
|      1|278356|
|      2|214903|
+-------+------+

Verified: every route belongs to exactly one CV fold (route-clean hyperparameter search).


## Shared pipeline stages (indexing, encoding, assembly, scaling)

In [14]:
label_indexer = StringIndexer(inputCol=TARGET_COLUMN, outputCol="label", handleInvalid="keep")

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_candidates
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_candidates
]

assembler_inputs = numeric_candidates + [f"{c}_ohe" for c in categorical_candidates]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features_raw", handleInvalid="keep")

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True)

shared_stages = [label_indexer] + indexers + encoders + [assembler, scaler]
print(f"Shared pipeline stages: {len(shared_stages)}")
print(f"Assembler input columns ({len(assembler_inputs)}): {assembler_inputs}")

Shared pipeline stages: 21
Assembler input columns (20): ['fare_publication_count', 'hour_of_day', 'is_first_stop', 'is_peak_hour', 'operator_routes', 'operator_trip_count', 'stop_activity', 'stop_progress_pct', 'stop_sequence', 'total_stops', 'unique_stops', 'fare_publication_level_ohe', 'fare_status_ohe', 'journey_type_ohe', 'operator_size_ohe', 'operator_workload_ohe', 'route_complexity_ohe', 'scheduled_time_ohe', 'stop_busyness_ohe', 'stop_position_ohe']


In [15]:
def with_weight_if_supported(estimator, weight_col="class_weight"):
    if estimator.hasParam("weightCol"):
        estimator = estimator.setWeightCol(weight_col)
        print(f"  {type(estimator).__name__}: weightCol enabled")
    else:
        print(f"  {type(estimator).__name__}: weightCol not supported by this estimator, skipping")
    return estimator

## Define four models, their param grids, and cross-validators

### Baseline: Logistic Regression (multinomial)

In [16]:
lr = LogisticRegression(featuresCol="features", labelCol="label", predictionCol="prediction", family="multinomial")
lr = with_weight_if_supported(lr)

lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.001, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

lr_pipeline = Pipeline(stages=shared_stages + [lr])

  LogisticRegression: weightCol enabled


### Decision Tree

In [17]:
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", predictionCol="prediction", seed=SEED)
dt = with_weight_if_supported(dt)

dt_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [8, 12, 18])
    .addGrid(dt.impurity, ["gini", "entropy"])
    .addGrid(dt.minInstancesPerNode, [1, 5])
    .build()
)

dt_pipeline = Pipeline(stages=shared_stages + [dt])

  DecisionTreeClassifier: weightCol enabled


### Random Forest

In [18]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    seed=SEED
)
rf = with_weight_if_supported(rf)

rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [100, 200])
    .addGrid(rf.maxDepth, [10, 15, 20])
    .build()
)

rf_pipeline = Pipeline(stages=shared_stages + [rf])

  RandomForestClassifier: weightCol enabled


In [19]:
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

def make_cv(pipeline, grid):
    return CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=grid,
        evaluator=f1_evaluator,
        numFolds=CV_FOLDS,
        foldCol="cv_fold",  # route-stratified folds assigned above -- replaces random row-level folds
        parallelism=2,
        seed=SEED,
    )

models = {
    "Logistic Regression (baseline)": make_cv(lr_pipeline, lr_grid),
    "Decision Tree":                  make_cv(dt_pipeline, dt_grid),
    "Random Forest":                  make_cv(rf_pipeline, rf_grid),
}
print(f"{len(models)} models configured for cross-validated training:")
for name in models:
    print(f"  - {name}")

3 models configured for cross-validated training:
  - Logistic Regression (baseline)
  - Decision Tree
  - Random Forest


## Train, tune, and time each model

In [20]:
fitted_models = {}
training_times = {}

for name, cv in models.items():
    print(f"\nTraining: {name} ...")
    start = time.time() 
    fitted_models[name] = cv.fit(train_df)
    elapsed = time.time() - start
    training_times[name] = elapsed
    print(f"  done in {elapsed:.1f}s -- best CV F1: {max(fitted_models[name].avgMetrics):.4f}")


Training: Logistic Regression (baseline) ...
  done in 1131.6s -- best CV F1: 0.5294

Training: Decision Tree ...
  done in 2550.4s -- best CV F1: 0.4589

Training: Random Forest ...
  done in 17298.8s -- best CV F1: 0.4964


## Evaluate

In [21]:
metric_names = ["accuracy", "weightedPrecision", "weightedRecall", "f1"]
results = []

predictions_by_model = {}
for name, cv_model in fitted_models.items():
    preds = cv_model.transform(test_df).cache()
    predictions_by_model[name] = preds

    row = {"Model": name, "Training Time (s)": round(training_times[name], 1)}
    for metric in metric_names:
        evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=metric)
        row[metric] = round(evaluator.evaluate(preds), 4)
    results.append(row)

comparison_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print(comparison_df.to_string(index=False))

                         Model  Training Time (s)  accuracy  weightedPrecision  weightedRecall     f1
                 Random Forest            17298.8    0.6998             0.7007          0.6998 0.6991
                 Decision Tree             2550.4    0.6056             0.6002          0.6056 0.5932
Logistic Regression (baseline)             1131.6    0.5665             0.5773          0.5665 0.5707


### Confusion matrix per model

In [22]:
label_model = fitted_models[list(fitted_models)[0]].bestModel.stages[0]
label_names = label_model.labels
print("Label index -> class name:", dict(enumerate(label_names)))

for name, preds in predictions_by_model.items():
    print(f"\n--- Confusion matrix: {name} ---")
    (
        preds.groupBy("label", "prediction")
        .count()
        .orderBy("label", "prediction")
        .show(n_classes * n_classes)
    )

Label index -> class name: {0: 'High', 1: 'Medium', 2: 'Low'}

--- Confusion matrix: Logistic Regression (baseline) ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|35258|
|  0.0|       1.0|27130|
|  0.0|       2.0| 6185|
|  1.0|       0.0|22023|
|  1.0|       1.0|26829|
|  1.0|       2.0| 9639|
|  2.0|       0.0| 5296|
|  2.0|       1.0|13632|
|  2.0|       2.0|47563|
+-----+----------+-----+


--- Confusion matrix: Decision Tree ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|27096|
|  0.0|       1.0|22538|
|  0.0|       2.0|18939|
|  1.0|       0.0|16151|
|  1.0|       1.0|34883|
|  1.0|       2.0| 7457|
|  2.0|       0.0| 3164|
|  2.0|       1.0| 8093|
|  2.0|       2.0|55234|
+-----+----------+-----+


--- Confusion matrix: Random Forest ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|46644|
|  0.0|       1.0| 2886|
|  0.0|       2.0|190

### Feature importance

In [23]:
sample_transformed = fitted_models["Decision Tree"].bestModel.transform(train_df.limit(5))
metadata = sample_transformed.schema["features_raw"].metadata
expanded_feature_names = []
attrs = metadata["ml_attr"]["attrs"]

for attr_type in ["numeric", "binary", "nominal"]:
    if attr_type in attrs:
        expanded_feature_names.extend(
            [item["name"] for item in attrs[attr_type]]
        )
print("Expanded feature names:", len(expanded_feature_names))
print(expanded_feature_names[:20])
n_importances = len(fitted_models["Decision Tree"].bestModel.stages[-1].featureImportances)
print(f"assembler_inputs (raw columns)          : {len(assembler_inputs)}")
print(f"expanded_feature_names (vector dimensions): {len(expanded_feature_names)}")
print(f"featureImportances length                : {n_importances}")
assert len(expanded_feature_names) == n_importances, (
    "expanded_feature_names length doesn't match featureImportances length "
    "-- check that all fitted pipelines share the same indexer/encoder/assembler stages."
)

Expanded feature names: 1230
['fare_publication_count', 'hour_of_day', 'is_first_stop', 'is_peak_hour', 'operator_routes', 'operator_trip_count', 'stop_activity', 'stop_progress_pct', 'stop_sequence', 'total_stops', 'unique_stops', 'fare_publication_level_ohe_High', 'fare_publication_level_ohe_Low', 'fare_status_ohe_Available', 'fare_status_ohe_Unavailable', 'journey_type_ohe_Medium', 'journey_type_ohe_Long', 'journey_type_ohe_Short', 'operator_size_ohe_Large', 'operator_size_ohe_Medium']
assembler_inputs (raw columns)          : 20
expanded_feature_names (vector dimensions): 1230
featureImportances length                : 1230


In [24]:
for name in ["Decision Tree", "Random Forest"]:
    best_pipeline_model = fitted_models[name].bestModel
    classifier_stage = best_pipeline_model.stages[-1]
    importances = classifier_stage.featureImportances

    imp_df = pd.DataFrame({
        "feature_name": expanded_feature_names,
        "importance": importances.toArray(),
    }).sort_values("importance", ascending=False).head(15)
    print(f"\nTop 15 features -- {name}:")
    print(imp_df[["feature_name", "importance"]].to_string(index=False))

lr_best_model = fitted_models["Logistic Regression (baseline)"].bestModel
lr_stage = lr_best_model.stages[-1]
coef_matrix = lr_stage.coefficientMatrix.toArray()  # shape: (n_classes, n_features)
print(f"\nLogistic Regression coefficient magnitudes (baseline), by class ({label_names}):")
for class_idx, class_name in enumerate(label_names):
    coef_df = pd.DataFrame({
        "feature_name": expanded_feature_names,
        "coefficient": coef_matrix[class_idx],
    })
    coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
    top = coef_df.sort_values("abs_coefficient", ascending=False).head(10)
    print(f"\n  Class '{class_name}' -- top 10 by |coefficient|:")
    print("  " + top[["feature_name", "coefficient"]].to_string(index=False).replace("\n", "\n  "))


Top 15 features -- Decision Tree:
               feature_name  importance
               unique_stops    0.458319
              stop_activity    0.398423
                total_stops    0.050687
            operator_routes    0.047285
        operator_trip_count    0.016751
   stop_busyness_ohe_Medium    0.007959
   route_complexity_ohe_Low    0.006855
   operator_size_ohe_Medium    0.005248
                hour_of_day    0.002840
    operator_size_ohe_Small    0.002681
route_complexity_ohe_Medium    0.000911
   stop_position_ohe_Middle    0.000740
              is_first_stop    0.000308
          stop_progress_pct    0.000153
scheduled_time_ohe_06:41:00    0.000148

Top 15 features -- Random Forest:
               feature_name  importance
              stop_activity    0.184675
               unique_stops    0.138184
      stop_busyness_ohe_Low    0.092159
                total_stops    0.081273
     stop_busyness_ohe_High    0.062307
route_complexity_ohe_Medium    0.049358
   route_c

### Cross-validation results 

In [25]:
for name, cv_model in fitted_models.items():
    print(f"\n--- CV results: {name} ---")
    for params, metric in zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics):
        param_str = ", ".join(f"{p.name}={v}" for p, v in params.items())
        print(f"  F1={metric:.4f}  [{param_str}]")
    print(f"  >> best F1: {max(cv_model.avgMetrics):.4f}")


--- CV results: Logistic Regression (baseline) ---
  F1=0.5278  [regParam=0.001, elasticNetParam=0.0]
  F1=0.5294  [regParam=0.001, elasticNetParam=0.5]
  F1=0.5182  [regParam=0.01, elasticNetParam=0.0]
  F1=0.5114  [regParam=0.01, elasticNetParam=0.5]
  F1=0.5111  [regParam=0.1, elasticNetParam=0.0]
  F1=0.4580  [regParam=0.1, elasticNetParam=0.5]
  >> best F1: 0.5294

--- CV results: Decision Tree ---
  F1=0.4588  [maxDepth=8, impurity=gini, minInstancesPerNode=1]
  F1=0.4589  [maxDepth=8, impurity=gini, minInstancesPerNode=5]
  F1=0.3946  [maxDepth=8, impurity=entropy, minInstancesPerNode=1]
  F1=0.3945  [maxDepth=8, impurity=entropy, minInstancesPerNode=5]
  F1=0.3967  [maxDepth=12, impurity=gini, minInstancesPerNode=1]
  F1=0.3972  [maxDepth=12, impurity=gini, minInstancesPerNode=5]
  F1=0.3453  [maxDepth=12, impurity=entropy, minInstancesPerNode=1]
  F1=0.3453  [maxDepth=12, impurity=entropy, minInstancesPerNode=5]
  F1=0.3695  [maxDepth=18, impurity=gini, minInstancesPerNode=1]

### Cell 23 — Scalability notes (training time vs. model complexity)

In [26]:
scalability_df = comparison_df[["Model", "Training Time (s)", "f1"]].copy()
scalability_df["Time per 0.01 F1"] = (
    scalability_df["Training Time (s)"] / (scalability_df["f1"] * 100)
).round(2)
print(scalability_df.to_string(index=False))
print(
     "\n'Time per 0.01 F1' measures training efficiency. Lower values are "
    "better. Random Forest takes longer to train, so the extra cost should "
    "be justified by a higher F1 score."
)

                         Model  Training Time (s)     f1  Time per 0.01 F1
                 Random Forest            17298.8 0.6991            247.44
                 Decision Tree             2550.4 0.5932             42.99
Logistic Regression (baseline)             1131.6 0.5707             19.83

'Time per 0.01 F1' measures training efficiency. Lower values are better. Random Forest takes longer to train, so the extra cost should be justified by a higher F1 score.


### Model comparison table and best-model selection

In [27]:
print("="*70)
print("FINAL MODEL COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

best_model_name = comparison_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name].bestModel

baseline_row = comparison_df[comparison_df["Model"] == "Logistic Regression (baseline)"].iloc[0]
best_row = comparison_df.iloc[0]
uplift = best_row["f1"] - baseline_row["f1"]

print(f"\nBaseline (Logistic Regression) F1 : {baseline_row['f1']:.4f}")
print(f"Best model ({best_model_name}) F1{' ' * max(0, 20 - len(best_model_name))}: {best_row['f1']:.4f}")
print(f"Uplift over baseline               : {uplift:+.4f}")

print(f"\nSelected model: {best_model_name}")
print(
    "Best model selected based on F1 score, with accuracy, precision, recall, "
    "and training time also considered. Logistic Regression serves as the "
    "baseline, while Decision Tree and Random Forest are compared to determine "
    "whether their higher complexity provides a worthwhile performance gain."
)

FINAL MODEL COMPARISON
                         Model  Training Time (s)  accuracy  weightedPrecision  weightedRecall     f1
                 Random Forest            17298.8    0.6998             0.7007          0.6998 0.6991
                 Decision Tree             2550.4    0.6056             0.6002          0.6056 0.5932
Logistic Regression (baseline)             1131.6    0.5665             0.5773          0.5665 0.5707

Baseline (Logistic Regression) F1 : 0.5707
Best model (Random Forest) F1       : 0.6991
Uplift over baseline               : +0.1284

Selected model: Random Forest
Best model selected based on F1 score, with accuracy, precision, recall, and training time also considered. Logistic Regression serves as the baseline, while Decision Tree and Random Forest are compared to determine whether their higher complexity provides a worthwhile performance gain.


### Generalisation gap: CV F1 vs. held-out test F1

In [28]:
print("Generalisation gap (best CV F1 during tuning vs. held-out test F1):")
gap_rows = []
for name, cv_model in fitted_models.items():
    cv_f1 = max(cv_model.avgMetrics)
    test_f1 = comparison_df.loc[comparison_df["Model"] == name, "f1"].iloc[0]
    gap_rows.append({"Model": name, "CV F1": round(cv_f1, 4), "Test F1": round(test_f1, 4),
                      "Gap (CV - Test)": round(cv_f1 - test_f1, 4)})

gap_df = pd.DataFrame(gap_rows).sort_values("Gap (CV - Test)", ascending=False)
print(gap_df.to_string(index=False))
print("\nSmaller gap = better generalisation. This table is good evidence for your")
print("report's Evaluation / Critical Reflection section.")


Generalisation gap (best CV F1 during tuning vs. held-out test F1):
                         Model  CV F1  Test F1  Gap (CV - Test)
Logistic Regression (baseline) 0.5294   0.5707          -0.0413
                 Decision Tree 0.4589   0.5932          -0.1343
                 Random Forest 0.4964   0.6991          -0.2027

Smaller gap = better generalisation. This table is good evidence for your
report's Evaluation / Critical Reflection section.


### Save per-model test predictions 

In [30]:
EVAL_DIR = OUTPUT_DIR / "eval_predictions"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

for name, preds in predictions_by_model.items():
    out_cols = DISPLAY_ID_COLUMNS + [TARGET_COLUMN, "label", "prediction"]
    df_to_save = preds.select(*out_cols, *(
        [vector_to_array(F.col("probability")).alias("probability_array")]
        if "probability" in preds.columns else []
    ))
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "")
    path = EVAL_DIR / f"{safe_name}.parquet"
    df_to_save.write.mode("overwrite").parquet(str(path))
    print(f"Saved predictions -- {name}: {path}")

Saved predictions -- Logistic Regression (baseline): D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\eval_predictions\Logistic_Regression_baseline.parquet
Saved predictions -- Decision Tree: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\eval_predictions\Decision_Tree.parquet
Saved predictions -- Random Forest: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\eval_predictions\Random_Forest.parquet


### Save cross-validation results

In [31]:
cv_results = []
for name, cv_model in fitted_models.items():
    for params, metric in zip(cv_model.getEstimatorParamMaps(), cv_model.avgMetrics):
        cv_results.append({
            "model": name,
            "params": {p.name: v for p, v in params.items()},
            "avg_f1": metric,
        })

with open(OUTPUT_DIR / "cv_results.json", "w") as f:
    json.dump(cv_results, f, indent=2)
print(f"CV results saved to: {OUTPUT_DIR / 'cv_results.json'}")

CV results saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\cv_results.json


### Save feature importances / coefficients and label mapping

In [32]:
feature_importance_export = {
    "assembler_inputs": expanded_feature_names,  
    "label_names": label_names,
    "tree_importances": {},
    "lr_coefficients": {
        class_name: coef_matrix[i].tolist() for i, class_name in enumerate(label_names)
    },
}
for name in ["Decision Tree", "Random Forest"]:
    stage = fitted_models[name].bestModel.stages[-1]
    feature_importance_export["tree_importances"][name] = stage.featureImportances.toArray().tolist()

with open(OUTPUT_DIR / "feature_importance.json", "w") as f:
    json.dump(feature_importance_export, f, indent=2)
print(f"Feature importance / coefficients saved to: {OUTPUT_DIR / 'feature_importance.json'}")

Feature importance / coefficients saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\feature_importance.json


In [33]:
unseen_stand_in_path = OUTPUT_DIR / "unseen_records_sample.parquet"
test_df.drop("class_weight").write.mode("overwrite").parquet(str(unseen_stand_in_path))
print(f"Unseen-records stand-in saved to: {unseen_stand_in_path}")
print(f"Rows: {test_df.count():,}")

Unseen-records stand-in saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\unseen_records_sample.parquet
Rows: 193,555


## Persist best model

In [34]:
_fresh_check_f1 = round(f1_evaluator.evaluate(predictions_by_model[best_model_name]), 4)
_saved_f1 = round(float(comparison_df.loc[comparison_df["Model"] == best_model_name, "f1"].iloc[0]), 4)
assert abs(_fresh_check_f1 - _saved_f1) < 1e-6, (
    f"STALE DATA DETECTED: comparison_df says F1={_saved_f1} for {best_model_name}, "
    f"but predictions_by_model currently in memory gives F1={_fresh_check_f1}. "
    f"Do a clean kernel restart and Run All from the top before saving."
)
print(f"Integrity check passed: {best_model_name} F1={_fresh_check_f1} is consistent "
      f"with what is about to be saved.")

best_model.write().overwrite().save(BEST_MODEL_PATH)
print(f"Best model ({best_model_name}) saved to: {BEST_MODEL_PATH}")

comparison_df.to_csv(str(OUTPUT_DIR / "model_comparison.csv"), index=False)
print(f"Model comparison table saved to: {OUTPUT_DIR / 'model_comparison.csv'}")

Integrity check passed: Random Forest F1=0.6991 is consistent with what is about to be saved.
Best model (Random Forest) saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\best_model
Model comparison table saved to: D:\Fourth Semester\Big Data\Project\Timetable-Based Bus Route Performance Analytics\output\model_comparison.csv


In [ ]:
spark.stop()
print("Spark session stopped. Notebook 05 complete.")